In [3]:



year = 2022  # Replace with desired year
month = 8   # Replace with desired month

dest_path_1deg = '/Datastorage/saptarishi.dhanuka_asp25/era5_1deg'
dest_path_025deg = '/Datastorage/saptarishi.dhanuka_asp25/era5_025deg'

import zarr
import warnings
# Supress ECCodes warning
import xarray as xr
import numpy as np
import dask
import dask.distributed

import sys
sys.path.insert(1, '/home/saptarishi.dhanuka_asp25/weather/graphcast_dir/gc_dist')
import forecast.encabulator
import forecast.generate_model
import datetime


# Graphcast model to use as a template, defining important input and target variables
gc_checkpoint = '/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_0.25_37.npz'

# Remote URL for WeatherBench data
wb_url = 'gs://weatherbench2/datasets/era5/1959-2023_01_10-full_37-1h-0p25deg-chunk-1.zarr'

# Required for xarray/zarr; 'trust_env' ensures that they will properly use the environment-
# configured proxy server
storage_options = {'session_kwargs' : {'trust_env' : True}} 

# Load the Graphcast model and get required variables

(model_config, task_config, params) = forecast.generate_model.load_model(gc_checkpoint)
input_variables = list(task_config['input_variables'])
target_variables = list(task_config['target_variables'])
forcing_variables = list(task_config['forcing_variables'])

# Make the target repositories, if they don't already exist
import os
os.system(f'mkdir -vp {dest_path_025deg}')
os.system(f'mkdir -vp {dest_path_1deg}')

# Create a Dask Client to handle data download/processing with parallelism and automatic memory management
import dask.config
dask.config.set(
    {'distributed.worker.memory.target':False,
    'distributed.worker.memory.spill':False,}
)
# Suppress 'HTTP port already in use' warning
with warnings.catch_warnings(action="ignore"):
    Client = dask.distributed.Client(processes=False,threads_per_worker=20)

print("Opening WB dataset")
# Open the WeatherBench dataset, and set variables to chunk along only the time dimension
# Supress ECCodes warning
with warnings.catch_warnings(action="ignore"):
    ds = xr.open_dataset(wb_url,engine='zarr',storage_options=storage_options,chunks={'time':1,'latitude':-1,'longitude':-1,'level':-1})

Opening WB dataset


In [12]:
ds.isel(time=slice(-30, -1))

<xarray.Dataset> Size: 35GB
Dimensions:                                           (time: 29, latitude: 721,
                                                       longitude: 1440,
                                                       level: 37)
Coordinates:
  * latitude                                          (latitude) float32 3kB ...
  * level                                             (level) int64 296B 1 .....
  * longitude                                         (longitude) float32 6kB ...
  * time                                              (time) datetime64[ns] 232B ...
Data variables: (12/48)
    10m_u_component_of_wind                           (time, latitude, longitude) float32 120MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    10m_v_component_of_wind                           (time, latitude, longitude) float32 120MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    2m_dewpoint_temperature                           (time, latitude, longitude) float32 120MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    2m_temperature                                    (time, latitude, longitude) float32 120MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    angle_of_sub_gridscale_orography                  (latitude, longitude) float32 4MB dask.array<chunksize=(721, 1440), meta=np.ndarray>
    anisotropy_of_sub_gridscale_orography             (latitude, longitude) float32 4MB dask.array<chunksize=(721, 1440), meta=np.ndarray>
    ...                                                ...
    v_component_of_wind                               (time, level, latitude, longitude) float32 4GB dask.array<chunksize=(1, 37, 721, 1440), meta=np.ndarray>
    vertical_velocity                                 (time, level, latitude, longitude) float32 4GB dask.array<chunksize=(1, 37, 721, 1440), meta=np.ndarray>
    volumetric_soil_water_layer_1                     (time, latitude, longitude) float32 120MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    volumetric_soil_water_layer_2                     (time, latitude, longitude) float32 120MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    volumetric_soil_water_layer_3                     (time, latitude, longitude) float32 120MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    volumetric_soil_water_layer_4                     (time, latitude, longitude) float32 120MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>

In [ ]:

# Figure out which variables we can download from the dataset, based on variables the model needs and
# the variables the WB dataset has
model_vars = set(input_variables).union(set(target_variables)).union(set(forcing_variables))
download_vars = list(set.intersection(set(ds.data_vars),model_vars))

# Define the output variables including total precipitation, and set the proper encoding to compress them
output_vars = download_vars + ['total_precipitation_6hr']
compressed_vars = set(output_vars) - {'geopotential_at_surface','land_sea_mask','total_precipitation_6hr'} - set(forcing_variables)
encoding = {v : {'compressor' : forecast.encabulator.LayerQuantizer(nbits=16)} for v in compressed_vars}

print(f'Downloading: {year} / {month}')

# Define the start and end periods for the download, and define some handy timedeltas
dt_6h = datetime.timedelta(hours=6)
dt_1h = datetime.timedelta(hours=1)
date_start = datetime.datetime(year,month,1,0) # Start of the period
# Define the end of the period by subtracting 6h from the beginning of the next month
if (month == 12): # January is next month
    date_end = datetime.datetime(year+1,1,1,0) - dt_6h
else:
    date_end = datetime.datetime(year,month+1,1,0) - dt_6h

# Most of the output is just taken from the dataset, from the start to end every 6h.
# To slice by 6h increments, it seems easiest to use isel; slice with a timedelta stride errors.
output_ds = ds[download_vars].sel(time=slice(date_start,date_end)).isel(time=slice(None,None,6))

# Precipitation is a bit special, since it must be accumulated over the 6h period ending at
# the specified time.  Thus, we want to select hourly total_precipitation, beginning 5h before
# date_start and ending with date_end
ds_precip = ds['total_precipitation'].sel(time=slice(date_start-5*dt_1h,date_end))

# Group this data together using xr.goupby_bins
assert(ds_precip.time.size % 6 == 0)
ds_precip_grouped = ds_precip.groupby_bins('time',ds_precip.time.size//6)
ds_precip_sum = ds_precip_grouped.sum()

# Rename the 'time_bins' dimension to 'time', and reassign meaningful valid-time values
ds_precip_sum = ds_precip_sum.rename(time_bins='time')
ds_precip_sum['time'] = ds_precip['time'][5::6]
output_ds['total_precipitation_6hr'] = ds_precip_sum

# The Graphcast model expects latitude and longitude to be increasing, [-90 -> +90], so 
# it's better to perform any reordering here rather than every time the data is read
if output_ds.latitude.data[1] - output_ds.latitude.data[0] < 0:
    output_ds = output_ds.isel(latitude=slice(None,None,-1))
if output_ds.longitude.data[1] - output_ds.longitude.data[0] < 0:
    output_ds = output_ds.isel(longitude=slice(None,None,-1))

# Define output directories
out_zarr_025deg = f'{dest_path_025deg}/{year}/{month:02d}'
out_zarr_1deg = f'{dest_path_1deg}/{year}/{month:02d}'

# Subsample the quarter-degree output to give the 1-degree version
output_ds_1deg = output_ds.isel(latitude=slice(None,None,4)).isel(longitude=slice(None,None,4))

# Write the output; use compute=False to give dask the opportunity to mutually optimize the
# dataset operations
print(f'Writing to {out_zarr_025deg} and {out_zarr_1deg}')
tic = datetime.datetime.now()

# we comment this out since we don't want the 0.25 degree data yet
# delayed_025deg = output_ds.to_zarr(out_zarr_025deg,encoding=encoding,compute=False)
delayed_1deg = output_ds_1deg.to_zarr(out_zarr_1deg,encoding=encoding,compute=False)

(delayed_1deg) = dask.optimize(delayed_1deg)

# Import faulthanlder, which will act as a watchdog to dump a stacktrace in the event that things hang
import faulthandler
# Set the faulthandler to exit after half an hour, which should act as a failsafe to keep the downloads
# moving if one month stalls
faulthandler.dump_traceback_later(1800,exit=True)

try:
    Client.compute((delayed_1deg),sync=True)
except Exception as e:
    import sys
    print(f'Error downloading {year}-{month}, exception {e=}',file=sys.stderr)
    sys.exit(1)

toc = datetime.datetime.now()
print(f'{year}-{month} done in {(toc-tic).total_seconds():2}s')
